# PHATPSY 2.0 Tutorial: Exact-Integral Localized VB & Active-Site Embedding

This notebook demonstrates the **Phase 17** methodology across the **Stages 0–11** transition:

1. **Stage 3 Motivation (Non-Variational Diagnostic):** Measuring $E_{\text{exact}}[\Psi_{\text{PHATPSY}}]$ vs. the legacy single-center spherical model potential / regularized PK functional ($\text{N}_2$, $\text{CO}$, $\text{HF}$, $\text{LiH}$).
2. **Exact-Integral Diatomic `LocalizedVB`:** Variational two-center STO integrals and equilibrium PES scans (`H2` in `ADF-DZ` / `ADF-TZP`, `ControlConfig.localized_vb`).
3. **Polyatomic `MaskedVB` on $\mathrm{H}_2\mathrm{O}$:** Multi-center STO integrals with per-orbital AO localization domains (`strict_1c` vs. `bond_local`) recovering the bent $\approx 104^\circ$ equilibrium angle with **zero empirical $\text{H}$–$\text{H}$ parameters** ($> 95\%$ own-atom character).
4. **Active-Site $(S_{\mathcal{B}\mathcal{B}})^{-1}$ Slab Embedding (`CO/Ni(100)`):** Parameter-free (`pk_scale = 1.0`) Huzinaga/Phillips-Kleinman embedding in an 8-neighbor $C_{4v}$ $\mathrm{Ni}(100)$ slab (`lmin(S_eff_exact) > 0` vs. naive pairwise collapse `lmin < -1.5`) and **Blyholder $\sigma/\pi$ charge-transfer decomposition**.
5. **Hybrid Assembler & 3D Localized VB Orbitals:** Building `localized_vb` decks with `HybridAssembler.to_localized_vb_deck()` and rendering interactive 3D orbitals with `show_vb_orbitals()`.

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Configure repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "personal") else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from phatpsy import (
    AtomConfig,
    CalculationDeck,
    ControlConfig,
    HybridAssembler,
    PhatpsyDriver,
    ProlateGrid,
    build_diatomic,
    build_molecule,
    compute_vb_orbital_grid,
    evaluate_co_ni100_blyholder_benchmark,
    evaluate_diatomic_localized_vb_benchmark,
    evaluate_h2o_localized_vb_benchmark,
    interatomic_decomposition,
    run_localized_vb,
    show_vb_orbitals,
)
from phatpsy.vb_correction import occupied_spin_orbitals

HARTREE_TO_EV = 27.211386245988
print(f"PHATPSY 2.0 Localized VB environment ready (root={REPO_ROOT})")

---
## 1. Stage 3 Diagnostic: Why Active Covalent Bonds Need Exact Integrals

On the legacy single-center SCF path (`method="phatpsy_scf"`), PHATPSY uses spherical screening potentials $V_B^{\text{model}}(r_B)$ and regularized Phillips-Kleinman projectors for bonded atoms. Evaluating the **exact** Hamiltonian expectation value $E_{\text{exact}}[\Psi_{\text{PHATPSY}}]$ of PHATPSY's own localized Slater determinant reveals systematic non-variational over-binding across bonded molecules.

Phase 17 therefore relegates fitted model potentials and `pk_scale` regularization strictly to non-bonded spectator environments (`environment_embedding="model_potential"`), while active covalent bonds use `method="localized_vb"` / `"masked_vb"`.

In [ ]:
# Evaluate PHATPSY_SCF on H2 (DZP) vs. the exact energy of its own Slater determinant
driver = PhatpsyDriver(quiet=True)
deck_scf = build_diatomic("H", "H", bond_length=1.401, basis="DZP", spins=["up", "down"], seed_hybrids=True)
state_scf = driver.run_deck(deck_scf)

sos = occupied_spin_orbitals(state_scf)
X = np.asarray(state_scf["atom_coordinates"])
Z = np.asarray(state_scf["atom_charges"])[0]
grid = ProlateGrid(X[:, 0], X[:, 1], zeta_min=0.8, n_eta=48)
dec = interatomic_decomposition(grid, sos, float(Z[0]), float(Z[1]))

gap_ev = (float(state_scf.total_energy) - dec["E_det"]) * HARTREE_TO_EV
print(f"Legacy E_PHATPSY(H2, DZP) : {state_scf.total_energy:12.6f} Ha")
print(f"Exact E_det[Psi_PHATPSY]  : {dec['E_det']:12.6f} Ha (Hartree product={dec['E_HP']:.6f}, E_pauli={dec['E_pauli']:+.6f})")
print(f"Non-variational gap       : {gap_ev:+.3f} eV (-0.68 eV on H2 DZP; reaches -75 eV on N2 DZP)")

---
## 2. Exact-Integral Diatomic `LocalizedVB` (`ControlConfig.localized_vb`)

Switching to `ControlConfig.localized_vb(...)` sets `method="localized_vb"` and `spin_unrestricted=False`, dispatching directly to exact prolate-spheroidal two-center STO integrals and variational VB optimization.

In [ ]:
ctrl_h2 = ControlConfig.localized_vb(
    basis="ADF-DZ",
    domain_mode="bond_local",
    vb_wavefunction="cas",
    core_embedding="unprojected_eri",
)
deck_h2 = build_diatomic("H", "H", bond_length=1.427, basis="ADF-DZ", control=ctrl_h2, spin_unrestricted=False, net_spin=0)
state_h2 = run_localized_vb(deck_h2, quiet=True)
print(f"H2 LocalizedVB (ADF-DZ)  E = {state_h2['total_energy']:.6f} Ha | D_e = {state_h2['binding_energy_ev']:.3f} eV | -V/T = {state_h2['virial_ratio']:.4f}")

# Equilibrium PES scan & harmonic frequency benchmark
h2_bm = evaluate_diatomic_localized_vb_benchmark("H2", basis="ADF-DZ")
print(
    f"H2 Equilibrium (ADF-DZ)  : R_e = {h2_bm['r_e_bohr']:.4f} Bohr | "
    f"omega_e = {h2_bm['harmonic_frequency_cm']:.1f} cm^-1 | D_e = {h2_bm['binding_energy_ev']:.3f} eV"
)

---
## 3. Polyatomic `MaskedVB` on $\mathrm{H}_2\mathrm{O}$ (Zero Empirical $w_{\mathrm{HH}}$)

`MaskedVB` assigns each bonding or lone-pair orbital its own AO domain (`domain_mode="bond_local"` restricts each $\mathrm{O}$–$\mathrm{H}_i$ bond pair to $\{\mathrm{O}, \mathrm{H}_i\}$ while keeping Oxygen lone pairs on $\{\mathrm{O}\}$). Pauli repulsion between non-bonded $\mathrm{H}$ atoms emerges directly from exact multi-center integrals and determinant antisymmetry — no empirical $w_{\mathrm{HH}}$ calibration is used.

In [ ]:
h2o_bm = evaluate_h2o_localized_vb_benchmark(basis="DZ", domain_mode="bond_local")
print(f"H2O MaskedVB (DZ, bond_local):")
print(f"  Equilibrium theta_e : {h2o_bm['theta_e_deg']:.2f} deg (Expt: {h2o_bm['experimental'].get('bond_angle_deg', 104.5)} deg)")
print(f"  Dipole moment       : {h2o_bm['dipole_moment_debye']:.3f} D")
print(f"  Empirical w_HH      : {h2o_bm['empirical_w_hh']}")
print(f"  Own-atom character  : {h2o_bm['own_atom_character']}")

# Compare strict_1c -> bond_local -> delocalized regimes on a single bent H2O geometry
ctrl_cmp = ControlConfig.localized_vb(basis="DZ", domain_mode="bond_local")
deck_h2o = build_molecule(
    [("O", 0.0, 0.0, 0.0), ("H", 1.430, 0.0, 1.107), ("H", -1.430, 0.0, 1.107)],
    basis="DZ", control=ctrl_cmp, spin_unrestricted=False, spins=None, net_spin=0,
)
state_h2o = run_localized_vb(deck_h2o, quiet=True, compare_regimes=True)
print("Regime variational ordering (Ha):", state_h2o["regime_comparison"])

---
## 4. Active-Site Slab Embedding (`CO/Ni(100)`) & Blyholder $\sigma/\pi$ Decomposition

In an 8-neighbor $C_{4v}$ $\mathrm{Ni}(100)$ slab (`data/ni_surf_embedded.json`, `data/ni_sub_embedded.json`), naive pairwise PK summation (`sum_B S_AB S_BA`) destroys positive-definiteness of the projected metric ($\lambda_{\min}(\tilde{S}_{\text{naive}}) < -1.5$). PHATPSY 2.0's exact joint environment metric $(S_{\mathcal{B}\mathcal{B}})^{-1}$ (`environment_embedding="exact_joint_pk"`, `pk_scale=1.0`) keeps $\lambda_{\min}(\tilde{S}_{\text{exact}}) > +0.05$ without empirical damping and captures the substrate-enhanced **Blyholder $\pi$ back-donation dominance**.

In [ ]:
coni_bm = evaluate_co_ni100_blyholder_benchmark()
print("CO / Ni(100) Top-Site Chemisorption (R_NiC = 3.59 Bohr, DZP, pk_scale=1.0):")
print(f"  Pseudometric lmin(S_eff) : exact = {coni_bm['lmin_S_eff_exact']:+.5f} (> 0) | naive = {coni_bm['lmin_S_eff_naive']:+.5f} (< -1.5)")
print(f"  Regime E0 -> E1 -> E2    : {coni_bm['e_r0_hartree']:.6f} -> {coni_bm['e_r1_hartree']:.6f} -> {coni_bm['e_r2_hartree']:.6f} Ha")
print(f"  5sigma donation (dE_sig) : {coni_bm['delta_e_sigma_ev']:+.4f} eV")
print(f"  2pi* backbonding (dE_pi) : {coni_bm['delta_e_pi_ev']:+.4f} eV (pi share = {100.0 * coni_bm['pi_share']:.1f}%)")

---
## 5. `HybridAssembler.to_localized_vb_deck()` & 3D Localized VB Orbital Viewer

`HybridAssembler` exports closed-shell `localized_vb` decks with automatic active/spectator tagging and graph-derived `orbital_domains` (when `domain_mode="custom"`). Individual localized orbitals can be exported as Gaussian Cubes (`compute_vb_orbital_grid`) or inspected interactively (`show_vb_orbitals`).

In [ ]:
asm = HybridAssembler(root="O", hybridization="sp3_bent")
asm.cap_all("H")
deck_asm = asm.to_localized_vb_deck(basis="DZ", domain_mode="custom", active_atoms=["O1", "H1", "H2"])
print("Custom per-atom orbital_domains on O1:", deck_asm.geometry[0].orbital_domains)

grid_vb1 = compute_vb_orbital_grid(state_h2o, orbital_index=1, spacing=0.25)
print(f"H2O VB orbital 1 grid shape: {grid_vb1.dimensions}, max={grid_vb1.max_value:.4f}")

show_vb_orbitals(state_h2o, isovalue=0.05, height=550, title="H2O Localized VB Orbitals (Bond-Local MaskedVB)")